# Comparación de datos: réplicas LAB019-021 vs originales LAB004/006/009

**Alcance:** comparación *pura de datos de laboratorio* entre cada réplica (septiembre 2026) y su
original de abril 2026. **Sin modelo, sin theta, sin capa CO2** — ese análisis vive en notebooks posteriores.

| Par | Original (abril) | Réplica (sept) | Protocolo |
|---|---|---|---|
| 1 | LAB004 | LAB019 | A: 18°C → 21°C@99h → 16°C@171h; pulso 2 original t=51h |
| 2 | LAB006 | LAB020 | C arranque frío: 16°C → 18°C@123h → 21°C@219h; pulso 2 original t=75h |
| 3 | LAB009 | LAB021 | A: 18°C → 21°C@96h → 16°C@168h; pulso 2 original t=48h |

**Fuentes:** workbook homologado `mosto_natural_xthiol.xlsx` (hojas LAB004/006/009) para los
originales; para las réplicas: exports Y15 (EXPAuto/ONLINE), `Mediciones-Offline-LAB019-021.xlsx`
(densidad/Brix/horas de muestreo) y `Oculyze_report_LAB*.csv` (concentración, con resta de la fila CERO).

**Decisiones de diseño:**
- Dos alineaciones: por **reloj** (t desde el t0 de cada lote) y por **estado** (azúcar consumida), para separar diferencias de *velocidad* de diferencias de *forma*.
- La temperatura de la columna del Anton Paar del Excel offline **no se usa** (medición de T de muestra no confiable); el log de temperatura del biorreactor se incorporará en notebooks posteriores.
- La columna `Viability` del workbook homologado es un duplicado de la concentración Oculyze (corrupta): no se grafica; la viabilidad de las réplicas sí.
- Puntos con flag de QA se **marcan en rojo**, no se eliminan silenciosamente; la lista completa se exporta a `qa_flags.csv`.
- Nitrógeno en mg N/L: YAN = PAN + 0.823·NH3 (validado contra el canal YAN del propio Y15 en PI).

In [ ]:
from pathlib import Path
import csv
import re
import sys
from datetime import datetime, time as dtime

import matplotlib.pyplot as plt
from IPython import get_ipython
_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("matplotlib", "inline")
import numpy as np
import openpyxl
import pandas as pd
from IPython.display import display

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
ROOT = next(path for path in candidates if (path / "fermentation_model").exists())
FM = ROOT / "fermentation_model"

WORKBOOK_PATH = FM / "data" / "Laboratorio 2026" / "Vendimia_2026" / "mosto_natural_xthiol.xlsx"
REPLICA_DIR = FM / "data" / "mem2026" / "LAB019-021"
OFFLINE_PATH = REPLICA_DIR / "Mediciones-Offline-LAB019-021.xlsx"
Y15_PATHS = sorted(REPLICA_DIR.glob("EXPAuto*.txt")) + sorted(REPLICA_DIR.glob("ONLINE*.txt"))
OCULYZE_PATHS = {lab: REPLICA_DIR / f"Oculyze_report_{lab}.csv" for lab in ("LAB019", "LAB020", "LAB021")}

RESULTS_DIR = FM / "laboratory_2026" / "results" / "lab019_021_replica_vs_originals_data_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

required = [WORKBOOK_PATH, OFFLINE_PATH, *Y15_PATHS, *OCULYZE_PATHS.values()]
assert all(path.exists() for path in required), [str(path) for path in required if not path.exists()]
assert Y15_PATHS, "No se encontraron exportaciones Y15"

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 180, "font.size": 9.5, "axes.titlesize": 10.5})

COLOR_ORIG = "#4c72b0"
COLOR_REPL = "#dd8452"
COLOR_FLAG = "#d62728"

## Constantes de campaña (t0, eventos, umbrales)

- t0 de cada lote se lee de sus propios datos (fila t=0 del workbook; 21/09 17:00 = aplicación de levadura en las réplicas (confirmado por laboratorio; el pulso 1 se dosificó al mosto antes del inóculo, por lo que la muestra 01 es pre-inoculación)).
- Pulsos 2: originales con las horas documentadas (planificación de la réplica / `effective_nutrient_pulses`); réplicas con fecha-hora real desde la hoja `Nutricion` del Excel offline.
- Los cambios de setpoint son idénticos por diseño en cada par: se dibujan una sola vez (líneas punteadas).

In [ ]:
PAIRS = [
    ("LAB004", "LAB019", "Protocolo A (18->21@99h->16@171h)"),
    ("LAB006", "LAB020", "Protocolo C arranque frío (16->18@123h->21@219h)"),
    ("LAB009", "LAB021", "Protocolo A (18->21@96h->16@168h)"),
]
SP_EVENTS = {  # (t_h, setpoint C) — idénticos en original y réplica por diseño
    "LAB004": [(99, 21), (171, 16)],
    "LAB006": [(123, 18), (219, 21)],
    "LAB009": [(96, 21), (168, 16)],
}
PULSE_ORIG_H = {"LAB004": 51.0, "LAB006": 75.0, "LAB009": 48.0}
PULSE_REPL_DT = {  # hoja Nutricion del Excel offline, ejecutado real (disparo por densidad ~1040)
    "LAB019": datetime(2026, 9, 24, 9, 20),
    "LAB020": datetime(2026, 9, 24, 16, 40),
    "LAB021": datetime(2026, 9, 23, 16, 30),
}
T0_REPL = {lab: datetime(2026, 9, 21, 17, 0) for lab in PULSE_REPL_DT}  # aplicacion de levadura (confirmado laboratorio); no 16:00 del plan

GF_THRESHOLDS = [100.0, 90.0, 40.0]
DENSITY_THRESHOLD = 1040.0
NH3_THRESHOLD = 5.0
BRIX_VALID_MAX = 17.5  # Brix '22.5/22.6' en filas tardías del workbook = error de ingreso

## Carga de originales (workbook homologado)

In [ ]:
ORIG_COLS = {  # índice de columna en la hoja (0-based)
    "fecha": 1, "t": 2, "densidad": 4, "peso_seco": 6, "glucosa": 7, "fructosa": 8,
    "pan": 9, "nh3": 10, "yan": 11, "glicerol": 12, "piruvico": 13, "acetaldehido": 15,
    "etanol": 16, "brix": 36, "ocu_conc": 39,
}

def num(value):
    try:
        out = float(value)
        return out if np.isfinite(out) else np.nan
    except (TypeError, ValueError):
        return np.nan

ORIG, T0_ORIG = {}, {}
wb = openpyxl.load_workbook(WORKBOOK_PATH, read_only=True, data_only=True)
for lab in ("LAB004", "LAB006", "LAB009"):
    rows = []
    for r in wb[lab].iter_rows(min_row=2, values_only=True):
        t = num(r[ORIG_COLS["t"]])
        gf = num(r[ORIG_COLS["glucosa"]]) + num(r[ORIG_COLS["fructosa"]])
        dens = num(r[ORIG_COLS["densidad"]])
        if not np.isfinite(t) or (not np.isfinite(gf) and not np.isfinite(dens)):
            continue  # filas marcadoras vacías (t=68/99/41)
        stamp = pd.to_datetime(r[ORIG_COLS["fecha"]])
        if abs(t) < 1e-9:
            T0_ORIG[lab] = stamp.to_pydatetime()
        brix = num(r[ORIG_COLS["brix"]])
        rows.append({
            "t_h": t, "GF": gf, "YAN": num(r[ORIG_COLS["yan"]]), "NH3": num(r[ORIG_COLS["nh3"]]),
            "Gly": num(r[ORIG_COLS["glicerol"]]), "density": dens, "Brix": brix,
            "Brix_valid": brix if (np.isfinite(brix) and brix <= BRIX_VALID_MAX) else np.nan,
            "X_ocu": num(r[ORIG_COLS["ocu_conc"]]), "peso_seco": num(r[ORIG_COLS["peso_seco"]]),
            "etanol": num(r[ORIG_COLS["etanol"]]), "piruvico": num(r[ORIG_COLS["piruvico"]]),
            "acetaldehido": num(r[ORIG_COLS["acetaldehido"]]),
        })
    ORIG[lab] = pd.DataFrame(rows).set_index("t_h").sort_index()
wb.close()
assert set(T0_ORIG) == {"LAB004", "LAB006", "LAB009"}
for lab, df in ORIG.items():
    print(lab, "t0 =", T0_ORIG[lab], "| n:", {c: int(df[c].notna().sum()) for c in ("GF", "YAN", "density", "Brix_valid", "X_ocu")})

## Carga de réplicas (Excel offline + Y15 + Oculyze)

Fallbacks documentados:
- Filas de la muestra 11 del Excel aún sin metadatos: se asignan en orden F1/F2/F3 con fecha-hora 25/09 08:30 (hora de visita, corroborada por la descripción de Oculyze). **Flag.**
- Tiempos de observación = hora de *muestreo* del Excel (no la hora de medición del Y15); la muestra 1 de Oculyze usa su propio timestamp (17:21, muestra extra post-inoculación). **Flag.**
- La densidad "1070" de Oculyze muestra 1 (tres reactores, T=0) es un placeholder de la app: no se usa; la densidad maestra es la del Excel.
Las muestras offline se leen por sus metadatos reales. Las exportaciones Y15 presentes se incorporan automáticamente; las muestras sin Y15 conservan Brix/densidad y quedan marcadas en QA.

Para m12 se conservan los canales de azúcar crudos: los resultados negativos no se grafican como concentraciones. Cuando `GLU-FRU-320` es negativo, se usa `GLUCOSE-FRUCTOSE` si entrega un valor no negativo.

In [ ]:
MESES = {m: i + 1 for i, m in enumerate(
    ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
     "septiembre", "octubre", "noviembre", "diciembre"])}

def parse_oculyze_stamp(date_s, time_s):
    d = re.match(r"(\d{1,2}) de (\w+) de (\d{4})", date_s.strip())
    t = re.match(r"(\d{1,2}):(\d{2}):(\d{2}) (a|p)\. m\.", time_s.strip())
    assert d and t, (date_s, time_s)
    hora = int(t.group(1)) % 12 + (12 if t.group(4) == "p" else 0)
    return datetime(int(d.group(3)), MESES[d.group(2)], int(d.group(1)), hora, int(t.group(2)), int(t.group(3)))

def load_offline():
    wb = openpyxl.load_workbook(OFFLINE_PATH, read_only=True, data_only=True)
    rows = []
    for excel_row, r in enumerate(wb["MedicionesOffline"].iter_rows(min_row=2, values_only=True), start=2):
        fecha, hora, lab, brix, dens = r[0], r[1], r[2], r[5], r[6]
        if lab is None and brix is None and dens is None:
            continue
        if lab not in ("LAB019", "LAB020", "LAB021") or fecha is None or hora is None or r[4] is None:
            raise ValueError(f"MedicionesOffline fila {excel_row}: metadatos incompletos")
        hh = hora.time() if isinstance(hora, datetime) else hora
        rows.append({"lab": lab, "muestra": str(r[4]).strip(),
                     "stamp": datetime.combine(fecha.date() if hasattr(fecha, "date") else fecha, hh),
                     "Brix": num(brix), "density": num(dens)})
    wb.close()
    result = pd.DataFrame(rows)
    assert not result.duplicated(["lab", "muestra"]).any(), "muestras offline duplicadas"
    return result

def load_y15():
    frames = []
    for path in Y15_PATHS:
        for line in open(path, encoding="utf-8-sig", errors="replace"):
            p = line.rstrip("\n").split("\t")
            if len(p) < 6:
                continue
            val = float(p[3].replace(",", ".")) if re.fullmatch(r"-?\d+(,\d+)?", p[3].strip()) else np.nan
            frames.append({"sample": p[0], "ana": p[1], "val": val})
    df = pd.DataFrame(frames).drop_duplicates(["sample", "ana"], keep="first")
    return df.pivot(index="sample", columns="ana", values="val")

def load_oculyze():
    out = {}
    for lab, path in OCULYZE_PATHS.items():
        cero, rows = np.nan, []
        with open(path, encoding="utf-8-sig", newline="") as fh:
            for r in csv.DictReader(fh):
                if r["Name"].strip() == "CERO":
                    cero = num(r["Concentration"])
                    continue
                m = re.fullmatch(r"(LAB\d+)-(\d+)", r["Name"].strip())
                if not m:
                    continue
                rows.append({"lab": m.group(1), "n": int(m.group(2)),
                             "stamp_ocu": parse_oculyze_stamp(r["Date"], r["Time"]),
                             "X_ocu": num(r["Concentration"]) - cero,  # 'VALOR BASE RESTAR'
                             "viab": num(r["Viability"]), "budding": num(r["Budding Index"])})
        out[lab] = pd.DataFrame(rows)
    return out

OFFLINE = load_offline()
Y15 = load_y15()
OCU = load_oculyze()
print("offline:", OFFLINE.shape, "| Y15 pivot:", Y15.shape, "| oculyze:", {k: len(v) for k, v in OCU.items()})

In [ ]:
def replica_frame(lab):
    base = OFFLINE[OFFLINE["lab"] == lab].copy()
    base["t_h"] = [(s - T0_REPL[lab]).total_seconds() / 3600.0 for s in base["stamp"]]
    numbered = sorted((item for item in base["muestra"] if str(item).isdigit()), key=int)
    y15 = Y15.reindex([f"{lab}-PREH2O2"] + [f"{lab}-{i}" for i in numbered])
    y15.index = ["PI"] + numbered
    df = base.set_index("muestra").join(y15)
    nh3_raw = df.get("AMMONIA")
    yan = df.get("PAN") + 0.823 * nh3_raw.clip(lower=0.0)  # mismo censurado de NH3 que el holdout
    gf_320 = df.get("GLU-FRU-320")
    gf_low = df.get("GLUCOSE-FRUCTOSE")
    gf_320_valid = gf_320.where(gf_320 >= 0.0)
    gf_low_valid = gf_low.where(gf_low >= 0.0)
    gf = gf_320_valid.combine_first(gf_low_valid)
    gf_source = pd.Series("", index=df.index, dtype=object)
    gf_source.loc[gf_320_valid.notna()] = "GLU-FRU-320"
    gf_source.loc[gf_320_valid.isna() & gf_low_valid.notna()] = "GLUCOSE-FRUCTOSE"
    g_320 = df.get("GLUCOSE-320")
    g_low = df.get("GLUCOSE")
    glucose = g_320.where(g_320 >= 0.0).combine_first(g_low.where(g_low >= 0.0))
    out = pd.DataFrame({
        "GF": gf, "GF_source": gf_source, "GF_320_raw": gf_320, "GF_low_raw": gf_low,
        "G": glucose, "YAN_mgN": yan,
        "NH3": nh3_raw, "Gly": df.get("GLYCEROL"), "density": df["density"], "Brix": df["Brix"],
        "acetic": df.get("ACETIC ACID"), "piruvico": df.get("PYRUVIC ACID"),
        "acetaldehido": df.get("ACETALDEHID-1000"),
    })
    out["t_h"] = df["t_h"]
    ocu = OCU[lab].copy()
    ocu["t_h"] = [(s - T0_REPL[lab]).total_seconds() / 3600.0 for s in ocu["stamp_ocu"]]
    # hora de observación: muestreo del Excel para m2+; m1 usa su propio stamp (17:21, extra post-inoculación)
    excel_t = dict(zip(base["muestra"], base["t_h"]))
    ocu["t_h"] = [ocu_t if n == 1 else excel_t.get(str(n), ocu_t)
                  for n, ocu_t in zip(ocu["n"], ocu["t_h"])]
    return out, ocu.set_index("t_h").sort_index()[["X_ocu", "viab", "budding"]]

REPL, REPL_X = {}, {}
for lab in ("LAB019", "LAB020", "LAB021"):
    REPL[lab], REPL_X[lab] = replica_frame(lab)
    print(lab, "| n GF:", int(REPL[lab]["GF"].notna().sum()),
          "| t de", round(REPL[lab]["t_h"].min(), 1), "a", round(REPL[lab]["t_h"].max(), 1),
          "h | n X:", int(REPL_X[lab]["X_ocu"].notna().sum()),
          "| muestras sin Y15:", [i for i in REPL[lab].index if i != "PI" and REPL[lab].loc[i, ["GF", "YAN_mgN", "Gly"]].isna().all()])

## Flags de QA (se marcan, no se borran)

In [ ]:
PULSE_REPL_H = {lab: (dt - T0_REPL[lab]).total_seconds() / 3600.0 for lab, dt in PULSE_REPL_DT.items()}

QA_FLAGS = [
    ("LAB004", "GF", "t=0", "GF0=145.9 con densidad 1067.7: 12 g/L bajo respecto a LAB019 a igual densidad; inconsistente", "marcado"),
    ("LAB004", "Brix", "t=146", "Brix 22.6 en vino casi seco = error de ingreso (regla Brix<=17.5)", "excluido por regla"),
    ("LAB006", "Brix", "t=146", "Brix 22.5 ídem", "excluido por regla"),
    ("LAB004/006/009", "Viability", "todo", "columna duplicada de la concentración Oculyze en el workbook, no es viabilidad", "no graficada"),
    ("LAB009", "X_ocu", "t=160", "concentración/viabilidad con crash final (6.8/4.6%) atípico", "marcado"),
    ("LAB004/006", "acetic", "todo", "canal 'Y15 Acetico' vacío en el workbook (sólo LAB009 tiene acético)", "panel secundario sin original"),
    ("LAB019", "GF", "m4", "GF 151.2 g/L por sobre el valor de m3 (no monótono); outlier Y15", "marcado"),
    ("LAB020", "G", "m4", "glucosa -7.28 g/L inválida", "canal G sin punto"),
    ("LAB019/020/021", "G", "m5-m7 (23-09)", "canal GLUCOSE-320 'Out' todo el miércoles 23/09", "sin split glu/fru ese día"),
    ("LAB021", "GF", "m8/m11", "GF faltante (m8) y 'Out High' (m11, glucosa 1.10)", "sin punto"),
    ("LAB019", "X_ocu", "m1", "28.65 medida 17:21 = 21 min post-inoculación (17:00): inóculo fresco sin mezclar del todo; plausible, no artefacto", "marcado"),
    ("LAB019/LAB021", "X_ocu", "m9", "concentración idéntica 72.12 en dos reactores: posible repetición en la app", "marcado"),
    ("LAB019/020/021", "density", "PI", "hora PI del Excel (12:30) pendiente de corrección a 15:10", "usada hora Excel"),
    ("LAB019/020/021", "T", "todo", "temperatura Anton Paar no confiable: excluida de este análisis", "canal excluido"),
]
for lab in ("LAB019", "LAB020", "LAB021"):
    for sample in REPL[lab].index:
        if sample != "PI" and REPL[lab].loc[sample, ["GF", "YAN_mgN", "Gly"]].isna().all():
            QA_FLAGS.append((lab, "Y15", f"m{sample}", "no existe exportación Y15 para esta muestra", "solo Brix/densidad"))
for lab in ("LAB019", "LAB020", "LAB021"):
    if "12" not in REPL[lab].index:
        continue
    row = REPL[lab].loc["12"]
    if pd.notna(row["GF_320_raw"]) and row["GF_320_raw"] < 0:
        action = "rango bajo usado" if row["GF_source"] == "GLUCOSE-FRUCTOSE" else "sin punto GF"
        QA_FLAGS.append((lab, "GF", "m12", "GLU-FRU-320 negativo; rango bajo auditado", action))
    if pd.isna(row["G"]):
        QA_FLAGS.append((lab, "G", "m12", "glucosa de rango bajo negativa", "sin punto G"))
flags_df = pd.DataFrame(QA_FLAGS, columns=["lote", "canal", "muestra", "problema", "acción"])
flags_df.to_csv(RESULTS_DIR / "qa_flags.csv", index=False, encoding="utf-8-sig")
display(flags_df)

# puntos que se marcan en rojo en las figuras: (lab, panel) -> [(t_h, valor)]
FLAG_MARKS = {}
def add_mark(lab, panel, t, v):
    if np.isfinite(t) and np.isfinite(v):
        FLAG_MARKS.setdefault((lab, panel), []).append((float(t), float(v)))

add_mark("LAB004", "GF", ORIG["LAB004"].index[0], ORIG["LAB004"]["GF"].iloc[0])
add_mark("LAB019", "GF", REPL["LAB019"].loc["4", "t_h"], REPL["LAB019"].loc["4", "GF"])
for lab in ("LAB019", "LAB021"):  # X: m1 (slug) y m9 (duplicado 72.12)
    xf = REPL_X[lab]
    for n_row, ocu in zip(OCU[lab]["n"], xf.itertuples()):
        if n_row in (1, 9):
            add_mark(lab, "X", ocu.Index, ocu.X_ocu)
lab009x = ORIG["LAB009"]
add_mark("LAB009", "X", lab009x.index[-1], lab009x["X_ocu"].iloc[-1])

## Hitos por lote

Cruce de densidad 1040, tiempo a GF=100/90/40 y agotamiento de NH3 (primera vez ≤ 5 mg/L),
interpolados en cada serie. `NaN` = aún no alcanzado (campaña en curso).

In [ ]:
def first_crossing(time, values, threshold, direction="below"):
    t = np.asarray(time, float); v = np.asarray(values, float)
    ok = np.isfinite(t) & np.isfinite(v)
    t, v = t[ok], v[ok]
    if len(v) < 2:
        return np.nan
    for i in range(len(v) - 1):
        v0, v1 = v[i], v[i + 1]
        hit = (v0 > threshold >= v1) if direction == "below" else (v0 < threshold <= v1)
        if hit:
            frac = 0.0 if v1 == v0 else (threshold - v0) / (v1 - v0)
            return t[i] + frac * (t[i + 1] - t[i])
    return np.nan

MILESTONE_DEFS = [
    ("t_cruce_dens1040_h", "dens 1040"), ("t_GF100_h", "GF 100"), ("t_GF90_h", "GF 90"),
    ("t_GF40_h", "GF 40"), ("t_NH3_agotado_h", "NH3 <= 5"),
]

def milestone_row(lab, is_replica):
    df = REPL[lab] if is_replica else ORIG[lab]
    t = df["t_h"] if is_replica else df.index
    return {"lote": lab,
            "t_cruce_dens1040_h": first_crossing(t, df["density"], DENSITY_THRESHOLD),
            "t_GF100_h": first_crossing(t, df["GF"], 100.0),
            "t_GF90_h": first_crossing(t, df["GF"], 90.0),
            "t_GF40_h": first_crossing(t, df["GF"], 40.0),
            "t_NH3_agotado_h": first_crossing(t, df["NH3"], NH3_THRESHOLD)}

rows = []
for orig, repl, _ in PAIRS:
    rows.append(milestone_row(orig, False))
    r = milestone_row(repl, True)
    for col, _short in MILESTONE_DEFS:
        d = r[col] - rows[-1][col]
        r["delta_" + col] = d if np.isfinite(d) else np.nan
    rows.append(r)
milestones_df = pd.DataFrame(rows).set_index("lote")
milestones_df.round(2).to_csv(RESULTS_DIR / "milestones.csv", encoding="utf-8-sig")
display(milestones_df.round(2))

## Figuras por par — alineación por reloj

Paneles: GF, nitrógeno (YAN), glicerol, densidad, Brix y concentración Oculyze (CERO restado en
réplicas). Líneas punteadas = cambios de setpoint (idénticos por diseño); rayado = pulso 2 del
original (azul) y de la réplica (naranja). Puntos con borde rojo = QA flag.

In [ ]:
PANELS = [
    ("GF", "GF (g/L)"), ("YAN", "YAN (mg N/L)"), ("Gly", "Glicerol (g/L)"),
    ("density", "Densidad (g/L)"), ("Brix", "Brix (-)"), ("X", "Conc. Oculyze (unid.)"),
]

def serie(lab, panel):
    if lab in ORIG:
        df = ORIG[lab]
        col = {"YAN": "YAN", "X": "X_ocu", "Brix": "Brix_valid"}.get(panel, panel)
        return df.index.to_numpy(float), df[col].to_numpy(float)
    df = REPL[lab]
    if panel == "YAN":
        v = df["YAN_mgN"]
    elif panel == "X":
        xf = REPL_X[lab]
        return xf.index.to_numpy(float), xf["X_ocu"].to_numpy(float)
    else:
        v = df[panel]
    return df["t_h"].to_numpy(float), v.to_numpy(float)

for orig, repl, proto in PAIRS:
    fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.6), sharex=True)
    for ax, (panel, title) in zip(axes.flat, PANELS):
        for lab, color, label in ((orig, COLOR_ORIG, "original"), (repl, COLOR_REPL, "réplica")):
            t, v = serie(lab, panel)
            ax.plot(t, v, color=color, marker="o", ms=3.5, lw=1.1, label=label)
            for tm, vm in FLAG_MARKS.get((lab, panel), []):
                ax.plot(tm, vm, "o", mfc="none", mec=COLOR_FLAG, ms=9, mew=1.6)
        if panel == "GF":
            ax.axvline(PULSE_ORIG_H[orig], color=COLOR_ORIG, ls="--", lw=1.0, alpha=0.55, label="pulso 2 original")
            ax.axvline(PULSE_REPL_H[repl], color=COLOR_REPL, ls="--", lw=1.0, alpha=0.55, label="pulso 2 réplica")
        for t_sp, _ in SP_EVENTS[orig]:
            ax.axvline(t_sp, color="k", ls=":", lw=0.9, alpha=0.45)
        ax.set_title(title)
        ax.set_xlabel("t desde t0 (h)")
    axes.flat[0].legend(loc="best", frameon=True, fontsize=8.5)
    fig.suptitle(f"{orig} (abril) vs {repl} (septiembre) — {proto} — alineación por reloj", fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    display(fig)
    plt.close(fig)
print("figura mostrada arriba (inline)")

## Comparación conjunta por reloj

Los tres pares en una sola figura, con las mismas señales de la alineación por estado.
El eje horizontal es el tiempo transcurrido desde el t0 de cada lote; los círculos rojos indican flags de QA.

In [ ]:
COMPARISON_PANELS = [("YAN", "YAN (mg N/L)"), ("NH3", "NH3 (mg/L)"), ("Gly", "Glicerol (g/L)"),
                     ("density", "Densidad (g/L)"), ("X", "Conc. Oculyze (unid.)")]

fig, axes = plt.subplots(len(PAIRS), len(COMPARISON_PANELS), figsize=(15.5, 3.2 * len(PAIRS)), sharex=True)
for row, (orig, repl, _proto) in enumerate(PAIRS):
    for ax, (panel, title) in zip(axes[row], COMPARISON_PANELS):
        for lab, color, label in ((orig, COLOR_ORIG, "original (abril)"), (repl, COLOR_REPL, "réplica (septiembre)")):
            t, v = serie(lab, panel)
            ax.plot(t, v, color=color, marker="o", ms=3.5, lw=1.1, label=label)
            for tm, vm in FLAG_MARKS.get((lab, panel), []):
                ax.plot(tm, vm, "o", mfc="none", mec=COLOR_FLAG, ms=9, mew=1.6)
        ax.set_title(f"{orig} vs {repl}: {title}", fontsize=9.5)
        if row == len(PAIRS) - 1:
            ax.set_xlabel("t desde t0 (h)")
axes[0, 0].legend(loc="best", frameon=True, fontsize=7.5)
fig.suptitle("Alineación por reloj (t desde t0 de cada lote)", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.97))
display(fig)
plt.close(fig)
print("figura mostrada arriba (inline)")

## Alineación por estado (azúcar consumida)

Mismas señales contra C = GF0 − GF de cada lote: elimina el corrimiento temporal y compara la
*forma* (p. ej., la meseta GF≈40–45 de LAB021 contra la de LAB009 queda visible a igual progreso).
Nota: el GF0 de LAB004 está marcado como inconsistente, por lo que su C arrastra ese offset.

In [ ]:
def consumed(lab):
    df = ORIG[lab] if lab in ORIG else REPL[lab]
    gf = df["GF"].to_numpy(float)
    gf0 = np.nanmax(gf[:1]) if lab in ORIG else REPL[lab]["GF"].dropna().iloc[0]
    t = df.index.to_numpy(float) if lab in ORIG else df["t_h"].to_numpy(float)
    return t, gf0 - gf, gf0

fig, axes = plt.subplots(len(PAIRS), len(COMPARISON_PANELS), figsize=(15.5, 3.2 * len(PAIRS)), sharex=True)
for row, (orig, repl, _proto) in enumerate(PAIRS):
    for ax, (panel, title) in zip(axes[row], COMPARISON_PANELS):
        for lab, color in ((orig, COLOR_ORIG), (repl, COLOR_REPL)):
            if panel == "X" and lab in REPL:
                xf = REPL_X[lab]
                df = REPL[lab]
                gf = df["GF"].to_numpy(float); t_gf = df["t_h"].to_numpy(float)
                ok = np.isfinite(gf)
                gf0 = np.nanmax(gf[ok][:1])
                t_x = xf.index.to_numpy(float)
                c_x = np.interp(t_x, t_gf[ok], gf0 - gf[ok])
                ax.plot(c_x, xf["X_ocu"].to_numpy(float), color=color, marker="o", ms=3.5, lw=1.1)
                continue
            t, c, gf0 = consumed(lab)
            df = ORIG[lab] if lab in ORIG else REPL[lab]
            if panel == "YAN":
                v = df["YAN"].to_numpy(float) if lab in ORIG else df["YAN_mgN"].to_numpy(float)
            elif panel == "NH3":
                v = df["NH3"].to_numpy(float)
            else:
                col = "X_ocu" if panel == "X" else panel
                v = df[col].to_numpy(float)
            ax.plot(c, v, color=color, marker="o", ms=3.5, lw=1.1)
        ax.set_title(f"{orig} vs {repl}: {title}", fontsize=9.5)
        if row == len(PAIRS) - 1:
            ax.set_xlabel("azúcar consumida GF0-GF (g/L)")
fig.suptitle("Alineación por estado (C = GF0 - GF de cada lote)", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.97))
display(fig)
plt.close(fig)
print("figura mostrada arriba (inline)")

## Hitos: original vs réplica (barras agrupadas con Δ)

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 5.2))
xg = np.arange(len(MILESTONE_DEFS)); width = 0.36
handles = []
for k, (orig, repl, _p) in enumerate(PAIRS):
    off = (k - 1) * (width + 0.03)
    vals_o = [milestones_df.loc[orig, c] for c, _s in MILESTONE_DEFS]
    vals_r = [milestones_df.loc[repl, c] for c, _s in MILESTONE_DEFS]
    b1 = ax.bar(xg + off - width / 2, vals_o, width, color=COLOR_ORIG, alpha=0.85)
    b2 = ax.bar(xg + off + width / 2, vals_r, width, color=COLOR_REPL, alpha=0.85)
    if k == 0:
        handles = [b1, b2]
    for xi, (vo, vr) in enumerate(zip(vals_o, vals_r)):
        if np.isfinite(vr):
            ax.annotate(f"Δ{vr - vo:+.0f}h", (xg[xi] + off + width / 2, vr), fontsize=7.5,
                        ha="center", va="bottom", rotation=90)
        else:
            ax.annotate("aún no", (xg[xi] + off + width / 2, 4), fontsize=7, ha="center",
                        rotation=90, color="gray")
ax.set_xticks(xg)
ax.set_xticklabels([s for _c, s in MILESTONE_DEFS])
ax.set_ylabel("t desde t0 (h)")
ax.legend(handles, ["original (abril)", "réplica (sept)"], title="pares 004/019 · 006/020 · 009/021", fontsize=8.5)
ax.set_title("Hitos de fermentación: original vs réplica (Δ = réplica - original)")
fig.tight_layout()
display(fig)
plt.close(fig)
print("figura mostrada arriba (inline)")

## Paneles secundarios (donde hay datos): pirúvico, acético, acetaldehído, etanol + viabilidad de réplicas

In [ ]:
fig, axes = plt.subplots(len(PAIRS), 4, figsize=(15.5, 3.0 * len(PAIRS)), sharex=True)
SEC = [("piruvico", "Pirúvico (mg/L)"), ("acetic", "Acético (g/L)"),
       ("acetaldehido", "Acetaldehído (mg/L)"), ("etanol", "Etanol (g/L, sólo originales)")]
for row, (orig, repl, _p) in enumerate(PAIRS):
    for ax, (panel, title) in zip(axes[row], SEC):
        for lab, color in ((orig, COLOR_ORIG), (repl, COLOR_REPL)):
            df = ORIG[lab] if lab in ORIG else REPL[lab]
            t = df.index.to_numpy(float) if lab in ORIG else df["t_h"].to_numpy(float)
            if panel not in df.columns:
                continue
            v = df[panel].to_numpy(float)
            if not np.isfinite(v).any():
                continue
            ax.plot(t, v, color=color, marker="o", ms=3.5, lw=1.1)
        ax.set_title(f"{orig} vs {repl}: {title}", fontsize=9.5)
        if row == len(PAIRS) - 1:
            ax.set_xlabel("t desde t0 (h)")
fig.suptitle("Paneles secundarios por par (azul = original, naranja = réplica)", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.96))
display(fig)
plt.close(fig)

fig2, axv = plt.subplots(figsize=(8.5, 3.6))
for lab, color in zip(("LAB019", "LAB020", "LAB021"), ("#4c72b0", "#55a868", "#c44e52")):
    xf = REPL_X[lab]
    axv.plot(xf.index, xf["viab"], color=color, marker="o", ms=3.5, lw=1.1, label=lab)
axv.set_xlabel("t desde t0 (h)")
axv.set_ylabel("Viabilidad Oculyze (%)")
axv.set_title("Viabilidad de las réplicas (original: columna corrupta en el workbook)")
axv.legend(fontsize=8.5)
fig2.tight_layout()
display(fig2)
plt.close(fig2)
print("figura mostrada arriba (inline)")

## Dispersión por par (grupo A vs C) y auditoría de entradas

In [ ]:
disp_rows = []
for orig, repl, _proto in PAIRS:
    grupo = "C (arranque frío)" if orig == "LAB006" else "A"
    for col, short in MILESTONE_DEFS:
        vo, vr = milestones_df.loc[orig, col], milestones_df.loc[repl, col]
        if np.isfinite(vo) and np.isfinite(vr):
            disp_rows.append({"par": f"{orig}->{repl}", "grupo": grupo, "hito": short,
                              "t_original_h": vo, "t_replica_h": vr, "delta_h": vr - vo})
disp_df = pd.DataFrame(disp_rows)
summary = disp_df.groupby("grupo")["delta_h"].agg(["count", "mean", "std", lambda s: s.abs().mean()])
summary.columns = ["n", "media Δ (h)", "std Δ (h)", "media |Δ| (h)"]
disp_df.round(2).to_csv(RESULTS_DIR / "dispersion_por_par.csv", index=False, encoding="utf-8-sig")
summary.round(2).to_csv(RESULTS_DIR / "dispersion_resumen.csv", encoding="utf-8-sig")
display(summary.round(2))
display(disp_df.round(2))

audit_rows = []
for lab in ("LAB004", "LAB006", "LAB009", "LAB019", "LAB020", "LAB021"):
    if lab in ORIG:
        df = ORIG[lab]
        audit_rows.append({"lote": lab, "campaña": "abril (original)",
                           "n_GF": int(df["GF"].notna().sum()), "n_YAN": int(df["YAN"].notna().sum()),
                           "n_densidad": int(df["density"].notna().sum()),
                           "n_Brix": int(df["Brix_valid"].notna().sum()),
                           "n_X_ocu": int(df["X_ocu"].notna().sum()),
                           "t_max_h": float(df.index.max())})
    else:
        df = REPL[lab]
        audit_rows.append({"lote": lab, "campaña": "sept (réplica)",
                           "n_GF": int(df["GF"].notna().sum()), "n_YAN": int(df["YAN_mgN"].notna().sum()),
                           "n_densidad": int(df["density"].notna().sum()),
                           "n_Brix": int(df["Brix"].notna().sum()),
                           "n_X_ocu": int(REPL_X[lab]["X_ocu"].notna().sum()),
                           "t_max_h": float(df["t_h"].max())})
audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(RESULTS_DIR / "input_audit.csv", index=False, encoding="utf-8-sig")
display(audit_df)

## Verificación de salidas

In [ ]:
csv_expected = [
    "qa_flags.csv", "milestones.csv", "dispersion_por_par.csv", "dispersion_resumen.csv", "input_audit.csv",
]
missing = [name for name in csv_expected if not (RESULTS_DIR / name).exists()]
assert not missing, missing
print("OK:", len(csv_expected), "CSV en", RESULTS_DIR.relative_to(FM), "| figuras solo inline en este notebook (sin PNG en results)")

## Notas de re-ejecución

- Re-ejecutar cuando llegue la muestra 16 (lunes 28/09 08:30): los hitos "aún no" (GF40 de LAB019/020)
  y la cola de LAB021 se completan solos.
- El etanol del Alcolyzer y el log de temperatura del biorreactor entran en notebooks posteriores
  (holdout cinético y capa CO2); este notebook no los consume.